# YuE2 on Kaggle — native FP16 engine (revision 13, one cell, canvas first)

**First time:** run `yue2-build-kaggle-assets.ipynb` once on Kaggle with
Internet and **no GPU**, then save its outputs as a private dataset.
Add that dataset to this notebook with **Add Input**. You can also attach
the saved builder notebook's output directly. Nothing large needs to be
downloaded to your PC or uploaded from your PC.

**Every music session:** attach that saved input, select **GPU T4 ×2**
and enable Internet, then run the **single cell** below. It goes straight
through setup, weight verification and a persistent ComfyUI server, and
prints the ☁️ **Cloudflare link that opens the full node graph directly** —
exactly like your Movies / H3 canvases. No second cell to start, nothing
runs automatically. Both canvases are pre-installed under ComfyUI's
**Workflows** menu:

* **YuE2 Full Song** — Run Context → Load Engine → Style/Lyrics →
  Plan Score (optional) → Song Tokens → Acoustic Synthesis → VAE Decode
  → Save Song. Direct-to-audio by default; switch the Plan Score node to
  Melody or Full mode for an ABC score, or paste a score into a
  **YuE2 Custom ABC** node.
* **YuE2 Cover / Restyle** — upload your song with the standard
  **LoadAudio** node, analyze the melody (SheetSage2, FP32 for T4), then
  write the new style. The engine loads only after analysis finishes.

This restores the original
`yue2_infer 0.1.5` engine: FP16 transformer on GPU 0, CUDA-graph AR decoding
with SDPA, and tiled FP32 VAE on GPU 1. Default generation uses **Direct to
audio**, **16 Euler steps**, CFG **1.0**, and an **8-minute safety ceiling**.
The model chooses a natural ending from your prompt and lyrics. It does
not have to fill the ceiling. A token-limit cutoff is reported separately.
Every node executes its real stage against this **same FP16 engine**:
same precision, attention layout, token ceilings, solver defaults and
saved-run files (`status.json`, tokens, latents, masters, ZIP). One
**seed** on the Run Context node drives the whole song. ComfyUI's **Stop**
interrupts at a step boundary and saves earlier stages.

**The original Gradio generator is still inside.** Set `GRADIO_UI = True`
at the top of the cell and run it again to use the classic interface
instead; each route takes the engine lock from the other automatically
and a GPU-memory preflight runs before any model work. ComfyUI runs from
a session-only pinned checkout (v0.43.2) under the same constraints; your
asset dataset is not rebuilt and no model is downloaded again. If upstream
pins a new package your saved wheelhouse lacks, setup installs your saved
wheels first and fetches only the missing ones from PyPI automatically.

**No diagnostic song runs automatically.** Cover your own song in either
route by uploading its WAV/FLAC/MP3, pasting your original lyrics, and
analyzing the melody first. Extracted-score regeneration can change
timing, voice and arrangement; it is not exact audio-to-audio copying.
Optional **Transcribe lyrics too** uses Whisper when needed.

**T4 attention change:** equivalent grouped-head views make PyTorch's
memory-efficient kernel eligible, instead of the native T4 GQA math path.
The first supported CUDA dispatch is verified and logged. Unsupported
shapes retain native attention and are reported. CPU equivalence checks
passed; actual T4 speed improvement has not been measured.

Durations are maximums. A natural early ending or token cutoff is reported.
Long acoustic generations may span multiple chunks; listen at the reported
boundaries. Seven/eight-minute performance and continuity remain unverified.

The small installed environment uses session-only `/kaggle/temp/yue2_native`.
Large models stay in the attached read-only Kaggle input dataset.
Song audio, exact tokens and recovery latents use
`/kaggle/working/yue2_native/runs`. The paths may share underlying storage;
setup prints actual free space rather than assuming a separate 50 GB quota.
Old notebook caches and files are preserved.

**Process protection:** setup and launch stop only verified older engines
using this script/output root, including survivors of a kernel restart.
A process lock prevents two managed engines from loading together. GPU
memory is checked before loading and before generation. An unrecognized
GPU process is reported and left running; setup stops before generation.
CUDA OOM saves GPU process details and releases the failed engine's model.

**After generation, download the complete run ZIP in the wrapper you used,
or Save Version with outputs in Kaggle before ending the session.** A saved
draft or a notebook download containing logs does not preserve audio files.


In [ ]:
import base64, json, os, sys, zlib
from pathlib import Path
if "app" in globals():
    app.stop()  # Release a managed server from the earlier ComfyUI notebook.
if "native_process" in globals() and native_process.poll() is None:
    native_process.terminate()
    try:
        native_process.wait(timeout=20)
    except Exception:
        native_process.kill()
        native_process.wait(timeout=10)
# Close only the original YuE2 pipeline if a previous notebook left it
# inside this kernel. Other model variables/processes are not cleared.
legacy_pipe = globals().get("pipe")
if (legacy_pipe is not None and legacy_pipe.__class__.__module__ == "yue2.pipeline"
        and legacy_pipe.__class__.__name__ == "YuE2Pipeline"):
    if isinstance(globals().get("CANCEL"), dict):
        CANCEL["stop"] = True
    legacy_demo = globals().get("demo")
    if legacy_demo is not None and legacy_demo.__class__.__module__.startswith("gradio."):
        legacy_demo.close()
    legacy_pipe.close()
    globals().pop("pipe", None)
    import gc
    gc.collect()
    torch_module = sys.modules.get("torch")
    if torch_module is not None and torch_module.cuda.is_initialized():
        torch_module.cuda.empty_cache()
    print("Closed the previous in-kernel YuE2 pipeline; saved song files remain.")
support = Path("/kaggle/temp/yue2_native_support")
support.mkdir(parents=True, exist_ok=True)
payload = "..."
for relative, contents in json.loads(zlib.decompress(base64.b64decode(payload))).items():
    path = support / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(contents)
for name in list(sys.modules):
    if name.split(".")[0] in {"native_yue2", "kaggle_yue2"}:
        del sys.modules[name]
if str(support) not in sys.path:
    sys.path.insert(0, str(support))
from native_yue2.setup import install
ROOT = "/kaggle/temp/yue2_native"
OUTPUTS = "/kaggle/working/yue2_native/runs"
ASSET_DATASET = "auto"  # Attach saved YuE2 assets with Add Input. No download fallback.
native_python, native_env = install(ROOT, OUTPUTS, assets=ASSET_DATASET)

# --- choose your interface ----------------------------------------------------
GRADIO_UI = False      # False = ComfyUI canvas (default, Movies style). True = original Gradio generator.
ENABLE_CLOUDFLARE = True

import json as _json
import re as _re
import socket as _socket
import subprocess as _subprocess
import threading as _threading
import time as _time
import urllib.request as _urlrequest
from IPython.display import HTML, display
from native_yue2.setup import COMFY_PORT, install_comfyui, request_comfy_release
from native_yue2.tunnel import CloudflareAccess

support_path = Path(support)
engine_script = support_path / "native_yue2/engine.py"
runtime_dir = Path(OUTPUTS).parent / "runtime"
runtime_dir.mkdir(parents=True, exist_ok=True)


def _summarize_runs():
    for directory in sorted(Path(OUTPUTS).glob("*")):
        if directory.is_dir():
            status = directory / "status.json"
            note = (status.read_text(errors="replace")[:180] if status.exists() else "Partial run")
            print("run:", directory.name, "—", note, flush=True)
    for archive in sorted(Path(OUTPUTS).glob("*.zip")):
        print("zip:", archive.name, flush=True)


if GRADIO_UI:
    # --- original Gradio generator (opt-in) ------------------------------------
    request_comfy_release()  # A running ComfyUI canvas releases the engine; the lock hands over.
    if "cloudflare_access" in globals():
        cloudflare_access.close()
    cloudflare_access = CloudflareAccess(
        ROOT, assets=native_env.get("YUE2_ASSET_DATASET"),
        log=runtime_dir / "cloudflared.log")
    cloudflare_attempted = False
    log_path = runtime_dir / "engine.log"
    from native_yue2.process import start_engine

    native_process = start_engine(native_python, engine_script, native_env, ROOT, OUTPUTS)
    try:
        with log_path.open("a") as log:
            for line in native_process.stdout:
                log.write(line)
                log.flush()
                print(line, end="", flush=True)
                local = _re.search(r"Running on local URL:\s+http://127\.0\.0\.1:(\d+)", line)
                if ENABLE_CLOUDFLARE and local and not cloudflare_attempted:
                    cloudflare_attempted = True
                    try:
                        link = cloudflare_access.start(int(local.group(1)), env=native_env)
                        print(f"\nAdditional Cloudflare link: {link}", flush=True)
                        print("The Gradio share link remains available. Both use the same engine.", flush=True)
                    except Exception as exc:
                        print(f"Cloudflare is unavailable: {exc}\nThe Gradio link is still available.", flush=True)
        if native_process.wait() != 0:
            raise RuntimeError(f"Native engine stopped. Inspect {log_path}")
    finally:
        cloudflare_access.close()
        if native_process.poll() is None:
            native_process.terminate()
            try:
                native_process.wait(timeout=20)
            except _subprocess.TimeoutExpired:
                native_process.kill()
                native_process.wait(timeout=10)
else:
    # --- ComfyUI canvas (default; the Cloudflare link opens the graph directly) -
    def comfy_ready(port=COMFY_PORT):
        try:
            with _urlrequest.urlopen(f"http://127.0.0.1:{port}/system_stats", timeout=3) as response:
                return response.status == 200
        except Exception:
            return False

    if "comfy_tunnel" in globals() and comfy_tunnel is not None:
        comfy_tunnel.close()
    if "comfy_process" in globals() and comfy_process.poll() is None:
        comfy_process.terminate()
        try:
            comfy_process.wait(timeout=15)
        except _subprocess.TimeoutExpired:
            comfy_process.kill()
            comfy_process.wait(timeout=10)

    if comfy_ready():
        print("ComfyUI server already running; reusing it.", flush=True)
    else:
        with _socket.socket() as probe:
            if probe.connect_ex(("127.0.0.1", COMFY_PORT)) == 0:
                raise RuntimeError(f"Port {COMFY_PORT} is occupied by another program.")
        comfy_dir, comfy_env = install_comfyui(ROOT, OUTPUTS, native_env)
        log_path = runtime_dir / "comfyui.log"
        comfy_log = log_path.open("a")
        comfy_process = _subprocess.Popen(
            [native_python, "-u", "main.py", "--listen", "127.0.0.1", "--port", str(COMFY_PORT),
             "--output-directory", str(OUTPUTS), "--disable-auto-launch", "--preview-method", "none"],
            cwd=str(comfy_dir), env=comfy_env, stdout=_subprocess.PIPE,
            stderr=_subprocess.STDOUT, text=True, bufsize=1)

        def _drain(stream, log):
            for line in stream:
                log.write(line)
                log.flush()
                print(line, end="", flush=True)
            log.close()

        _threading.Thread(target=_drain, args=(comfy_process.stdout, comfy_log), daemon=True).start()
        for _ in range(150):
            if comfy_process.poll() is not None:
                raise RuntimeError("ComfyUI exited during startup; inspect the log above.")
            if comfy_ready():
                break
            _time.sleep(2)
        else:
            raise TimeoutError("ComfyUI did not become ready; inspect the log above.")

    object_info = _json.loads(_urlrequest.urlopen(
        f"http://127.0.0.1:{COMFY_PORT}/object_info", timeout=15).read().decode())
    missing = [name for name in ("YuE2RunContext", "YuE2LoadEngine", "YuE2SaveSong") if name not in object_info]
    if missing:
        raise RuntimeError(f"KaggleYuE2 nodes missing from the server: {missing}; inspect the startup log.")

    comfy_tunnel = CloudflareAccess(
        ROOT, assets=native_env.get("YUE2_ASSET_DATASET"),
        log=runtime_dir / "comfy-cloudflared.log")
    base_link = None
    if ENABLE_CLOUDFLARE:
        try:
            base_link = comfy_tunnel.start(COMFY_PORT, ready_path="/system_stats")
        except Exception as exc:
            print(f"Cloudflare is unavailable: {exc}\nThe local server keeps running; set GRADIO_UI = True for the Gradio route.", flush=True)
    print("\nIn ComfyUI: Workflows → YuE2 Full Song / YuE2 Cover · or press Run on the opened canvas", flush=True)
    if base_link:
        song_link = base_link + "/?yue2_workflow=song"
        cover_link = base_link + "/?yue2_workflow=cover"
        print("Full-song canvas: ", song_link, flush=True)
        print("Cover canvas:     ", cover_link, flush=True)
        display(HTML('<a href="' + song_link + '" target="_blank" rel="noopener noreferrer">Open the YuE2 full-song canvas</a><br>'
                     '<a href="' + cover_link + '" target="_blank" rel="noopener noreferrer">Open the YuE2 cover canvas</a><br>'
                     '<a href="' + base_link + '" target="_blank" rel="noopener noreferrer">Open plain ComfyUI</a>'))
    print("Saved songs so far:", flush=True)
    _summarize_runs()
    print("Keep this Kaggle session and this cell running while using ComfyUI.", flush=True)
    print("Generate from the canvas: Queue Prompt. Stop interrupts at a step boundary and saves earlier stages.", flush=True)

    try:
        while comfy_process.poll() is None:
            _time.sleep(5)
        raise RuntimeError("ComfyUI server stopped; inspect the log above.")
    finally:
        comfy_tunnel.close()


## Where your songs go

Each attempt has its own folder under `/kaggle/working/yue2_native/runs`,
whichever route produced it. After a successful generation the wrapper
offers WAV, FLAC, and a complete ZIP containing settings, tokens, timings
and recovery latents. Download it before stopping. The folder also appears
under Kaggle Output at `yue2_native/runs`. `status.json` records the
measured audio length and any cutoff; it does not establish musical
completeness. Listen to the ending and chunk boundaries. Files must be
downloaded or saved with a Kaggle output version before the session ends.

## Optional decode recovery (only if a run failed to export audio)

If audio decoding failed after `latent.npy` and `latent_manifest.json`
were saved, stop the UI cell, then run this in a **new** cell — it checks
the latent checksum and original model weights and decodes to a new
folder without regenerating tokens or latents:

```python
RECOVER_RUN = "/kaggle/working/yue2_native/runs/<folder-with-latent.npy>"
recovery_env = dict(native_env, YUE2_RECOVER_RUN=str(Path(RECOVER_RUN).resolve()))
import subprocess
subprocess.run([native_python, "-u", str(support / "native_yue2/engine.py")],
               env=recovery_env, check=True)
```

For a new session, attach the saved run ZIP as a Kaggle input and extract
it first; setup and the original model files from the attached dataset
are still required. ComfyUI artifacts from revision 2 use a different
format.

Provenance: rebuilt from the original Kaggle notebook preserved in Movies
commit `fcf4d45dc5dbca57b39e54311807410cdf41f737`.
Sources: [YuE2-3B](https://huggingface.co/m-a-p/YuE2-3B),
[YuE2 VAE](https://huggingface.co/m-a-p/YuE2-Vae).
The native inference wheel and model revisions are pinned; wheel and
model checksums are verified. Upstream model/license terms apply.
Source and checks are included in the Music repository for review.
